In [0]:
# ====================================================================
# ML MODEL 2: DEMAND FORECASTING
# ====================================================================
# Purpose: Predict future product sales (units) by category
#          for inventory planning and procurement decisions
# ====================================================================

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, 
    r2_score, mean_absolute_percentage_error
)
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
SILVER_SCHEMA = f"{PROJECT_NAME}_silver"
GOLD_SCHEMA = f"{PROJECT_NAME}_gold"

# MLflow experiment name
EXPERIMENT_NAME = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/retail-demand-forecast"

print("=" * 80)
print("📈 DEMAND FORECASTING MODEL")
print("=" * 80)
print(f"Experiment: {EXPERIMENT_NAME}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# MLFLOW EXPERIMENT SETUP
# ====================================================================
print("🔬 Setting up MLflow experiment...\n")

# Set or create experiment
mlflow.set_experiment(EXPERIMENT_NAME)

# Get experiment details
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"✅ Experiment ID: {experiment.experiment_id}")
print(f"✅ Experiment Location: {experiment.artifact_location}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# LOAD HISTORICAL SALES DATA
# ====================================================================
print("📂 Loading historical sales data...\n")

# Load monthly revenue data
monthly_revenue = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_revenue_monthly")

print(f"✅ Monthly revenue data loaded: {monthly_revenue.count():,} rows")

# Convert to Pandas for time series analysis
sales_df = monthly_revenue.toPandas()

# Ensure datetime type
sales_df['order_month'] = pd.to_datetime(sales_df['order_month'])

print(f"\nDate range: {sales_df['order_month'].min()} to {sales_df['order_month'].max()}")
print(f"Total months: {sales_df['order_month'].nunique()}")
print(f"Categories: {sales_df['product_category_english'].nunique()}")
print(f"States: {sales_df['customer_state'].nunique()}")

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# PREPARE TIME SERIES DATA
# ====================================================================
print("🔧 Preparing time series data...\n")

# Aggregate by month and category (simplified for this model)
monthly_category_sales = (sales_df
    .groupby(['order_month', 'product_category_english'])
    .agg({
        'total_items_sold': 'sum',
        'total_revenue': 'sum',
        'total_orders': 'sum',
        'unique_customers': 'sum',
        'avg_item_value': 'mean'
    })
    .reset_index()
    .sort_values(['product_category_english', 'order_month'])
)

print(f"✅ Aggregated data: {len(monthly_category_sales):,} records")
print(f"   Categories: {monthly_category_sales['product_category_english'].nunique()}")
print(f"   Months per category: {monthly_category_sales.groupby('product_category_english').size().mean():.1f}")

# Show sample
print("\nSample data:")
print(monthly_category_sales.head(10))

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# FEATURE ENGINEERING
# ====================================================================
print("🔧 Engineering time series features...\n")

def create_time_features(df, date_col='order_month'):
    """Create time-based features for forecasting"""
    df = df.copy()
    
    # Extract date components
    df['year'] = df[date_col].dt.year
    df['month'] = df[date_col].dt.month
    df['quarter'] = df[date_col].dt.quarter
    df['day_of_year'] = df[date_col].dt.dayofyear
    
    # Create cyclical features (month is cyclical: Dec → Jan)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    return df

# Apply feature engineering
forecast_df = create_time_features(monthly_category_sales)

# Create lag features (previous months' sales)
def create_lag_features(df, target_col='total_items_sold', lags=[1, 2, 3]):
    """Create lagged features per category"""
    df = df.copy()
    
    for lag in lags:
        df[f'{target_col}_lag_{lag}'] = (
            df.groupby('product_category_english')[target_col]
            .shift(lag)
        )
    
    return df

forecast_df = create_lag_features(forecast_df, 'total_items_sold', lags=[1, 2, 3])

# Create rolling features (moving averages)
def create_rolling_features(df, target_col='total_items_sold', windows=[3]):
    """Create rolling window features per category"""
    df = df.copy()
    
    for window in windows:
        df[f'{target_col}_rolling_mean_{window}'] = (
            df.groupby('product_category_english')[target_col]
            .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
        )
    
    return df

forecast_df = create_rolling_features(forecast_df, 'total_items_sold', windows=[3])

# Drop rows with NaN (from lag features)
forecast_df = forecast_df.dropna()

print(f"✅ Features created")
print(f"   Total features: {forecast_df.shape[1]}")
print(f"   Records after removing NaN: {len(forecast_df):,}")

print("\nFeature columns:")
print(forecast_df.columns.tolist())

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# PREPARE X AND Y
# ====================================================================
print("📊 Preparing features and target variable...\n")

# Target variable
target = 'total_items_sold'

# Feature columns
feature_columns = [
    'year',
    'month',
    'quarter',
    'month_sin',
    'month_cos',
    'total_items_sold_lag_1',
    'total_items_sold_lag_2',
    'total_items_sold_lag_3',
    'total_items_sold_rolling_mean_3',
    'total_revenue',
    'total_orders',
    'unique_customers',
    'avg_item_value'
]

# One-hot encode categories
forecast_df_encoded = pd.get_dummies(
    forecast_df, 
    columns=['product_category_english'],
    prefix='category',
    drop_first=False
)

# Get all category columns
category_columns = [col for col in forecast_df_encoded.columns if col.startswith('category_')]
all_features = feature_columns + category_columns

# Prepare X and y
X = forecast_df_encoded[all_features]
y = forecast_df_encoded[target]

print(f"✅ Features: {len(all_features)}")
print(f"   - Time features: {len(feature_columns)}")
print(f"   - Category features: {len(category_columns)}")
print(f"✅ Target: {target}")
print(f"✅ Samples: {len(X):,}")

print("\nTarget statistics:")
print(y.describe())

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# TRAIN/TEST SPLIT (TIME-BASED)
# ====================================================================
print("✂️ Splitting data (time-based)...\n")

# Time series split: use last 20% of time periods as test
split_point = int(len(forecast_df_encoded) * 0.8)

X_train = X.iloc[:split_point]
X_test = X.iloc[split_point:]
y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]

print(f"Training set: {len(X_train):,} samples")
print(f"  Date range: {forecast_df_encoded.iloc[:split_point]['order_month'].min()} to {forecast_df_encoded.iloc[:split_point]['order_month'].max()}")

print(f"\nTest set: {len(X_test):,} samples")
print(f"  Date range: {forecast_df_encoded.iloc[split_point:]['order_month'].min()} to {forecast_df_encoded.iloc[split_point:]['order_month'].max()}")

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# FEATURE SCALING
# ====================================================================
print("📏 Scaling features...\n")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# MODEL 1: RANDOM FOREST REGRESSOR
# ====================================================================
print("🌲 Training Random Forest Regressor...\n")

with mlflow.start_run(run_name="random_forest_demand") as run:
    
    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("target_variable", target)
    mlflow.log_param("num_features", len(all_features))
    
    # Train model
    rf_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
    
    rf_model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred_train = rf_model.predict(X_train_scaled)
    y_pred_test = rf_model.predict(X_test_scaled)
    
    # Calculate metrics
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    test_mape = mean_absolute_percentage_error(y_test, y_pred_test)
    
    # Log metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("test_mape", test_mape)
    
    # Print results
    print("✅ Random Forest Results:")
    print(f"  Train MAE:  {train_mae:.2f}")
    print(f"  Test MAE:   {test_mae:.2f}")
    print(f"  Train RMSE: {train_rmse:.2f}")
    print(f"  Test RMSE:  {test_rmse:.2f}")
    print(f"  Train R²:   {train_r2:.4f}")
    print(f"  Test R²:    {test_r2:.4f}")
    print(f"  Test MAPE:  {test_mape:.2%}")
    
    # Log model
    signature = infer_signature(X_train_scaled, rf_model.predict(X_train_scaled))
    mlflow.sklearn.log_model(
        rf_model,
        "random_forest_model",
        signature=signature,
        registered_model_name="retail_demand_random_forest"
    )
    
    rf_run_id = run.info.run_id
    print(f"\n✅ Model logged to MLflow (Run ID: {rf_run_id})")

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# MODEL 2: GRADIENT BOOSTING REGRESSOR
# ====================================================================
print("🚀 Training Gradient Boosting Regressor...\n")

with mlflow.start_run(run_name="gradient_boosting_demand") as run:
    
    # Log parameters
    mlflow.log_param("model_type", "GradientBoostingRegressor")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 7)
    mlflow.log_param("target_variable", target)
    
    # Train model
    gb_model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=7,
        random_state=42
    )
    
    gb_model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred_train = gb_model.predict(X_train_scaled)
    y_pred_test = gb_model.predict(X_test_scaled)
    
    # Calculate metrics
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    test_mape = mean_absolute_percentage_error(y_test, y_pred_test)
    
    # Log metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("test_mape", test_mape)
    
    # Print results
    print("✅ Gradient Boosting Results:")
    print(f"  Train MAE:  {train_mae:.2f}")
    print(f"  Test MAE:   {test_mae:.2f}")
    print(f"  Train RMSE: {train_rmse:.2f}")
    print(f"  Test RMSE:  {test_rmse:.2f}")
    print(f"  Train R²:   {train_r2:.4f}")
    print(f"  Test R²:    {test_r2:.4f}")
    print(f"  Test MAPE:  {test_mape:.2%}")
    
    # Log model
    signature = infer_signature(X_train_scaled, gb_model.predict(X_train_scaled))
    mlflow.sklearn.log_model(
        gb_model,
        "gradient_boosting_model",
        signature=signature,
        registered_model_name="retail_demand_gradient_boosting"
    )
    
    gb_run_id = run.info.run_id
    print(f"\n✅ Model logged to MLflow (Run ID: {gb_run_id})")

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# VISUALIZE PREDICTIONS
# ====================================================================
print("📊 Creating actual vs predicted plot...\n")

# Use Gradient Boosting predictions
y_pred_test_gb = gb_model.predict(X_test_scaled)

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred_test_gb
})

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(comparison_df['actual'], comparison_df['predicted'], alpha=0.5)
axes[0].plot([comparison_df['actual'].min(), comparison_df['actual'].max()], 
             [comparison_df['actual'].min(), comparison_df['actual'].max()], 
             'r--', lw=2)
axes[0].set_xlabel('Actual Sales')
axes[0].set_ylabel('Predicted Sales')
axes[0].set_title('Actual vs Predicted Sales')
axes[0].grid(alpha=0.3)

# Residual plot
residuals = comparison_df['actual'] - comparison_df['predicted']
axes[1].scatter(comparison_df['predicted'], residuals, alpha=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Sales')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')
axes[1].grid(alpha=0.3)

plt.tight_layout()

# Save and display
plt.savefig('/tmp/demand_forecast_predictions.png', dpi=150, bbox_inches='tight')
print("✅ Prediction plots saved")
display(plt.gcf())
plt.close()

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# FEATURE IMPORTANCE
# ====================================================================
print("📊 Analyzing feature importance...\n")

# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': all_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importance.head(15).to_string(index=False))

# Plot
plt.figure(figsize=(10, 8))
top_n = 15
plt.barh(feature_importance['feature'][:top_n], feature_importance['importance'][:top_n])
plt.xlabel('Importance')
plt.title(f'Top {top_n} Feature Importance (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()

# Save and display
plt.savefig('/tmp/demand_feature_importance.png', dpi=150, bbox_inches='tight')
print("\n✅ Feature importance plot saved")
display(plt.gcf())
plt.close()

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# FORECAST NEXT 3 MONTHS
# ====================================================================
print("🔮 Generating forecasts for next 3 months...\n")

# Get the latest date in our dataset
latest_date = forecast_df_encoded['order_month'].max()
print(f"Latest data date: {latest_date}")

# Create future dates (next 3 months)
future_dates = pd.date_range(
    start=latest_date + pd.DateOffset(months=1),
    periods=3,
    freq='MS'  # Month start
)

print(f"Forecasting for: {future_dates[0].date()} to {future_dates[-1].date()}")

# Get unique categories
categories = forecast_df['product_category_english'].unique()

future_predictions = []

for category in categories:
    # Get latest data for this category
    category_data = forecast_df_encoded[
        forecast_df_encoded[[col for col in forecast_df_encoded.columns if col.startswith('category_')]].sum(axis=1) > 0
    ].tail(3)  # Last 3 months for lag features
    
    for future_date in future_dates:
        # Create features for future date
        future_features = {
            'year': future_date.year,
            'month': future_date.month,
            'quarter': (future_date.month - 1) // 3 + 1,
            'month_sin': np.sin(2 * np.pi * future_date.month / 12),
            'month_cos': np.cos(2 * np.pi * future_date.month / 12),
        }
        
        # Use last known values for other features (simplified)
        if len(category_data) > 0:
            last_row = category_data.iloc[-1]
            future_features['total_revenue'] = last_row['total_revenue']
            future_features['total_orders'] = last_row['total_orders']
            future_features['unique_customers'] = last_row['unique_customers']
            future_features['avg_item_value'] = last_row['avg_item_value']
            future_features['total_items_sold_lag_1'] = last_row['total_items_sold']
            future_features['total_items_sold_lag_2'] = last_row.get('total_items_sold_lag_1', last_row['total_items_sold'])
            future_features['total_items_sold_lag_3'] = last_row.get('total_items_sold_lag_2', last_row['total_items_sold'])
            future_features['total_items_sold_rolling_mean_3'] = last_row.get('total_items_sold_rolling_mean_3', last_row['total_items_sold'])
        
        # Add category encoding
        for cat_col in category_columns:
            cat_name = cat_col.replace('category_', '')
            future_features[cat_col] = 1 if cat_name == category else 0
        
        future_predictions.append({
            'forecast_date': future_date,
            'product_category': category,
            'features': future_features
        })

print(f"✅ Created {len(future_predictions)} forecast records")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# GENERATE PREDICTIONS
# ====================================================================
print("🎯 Making demand predictions...\n")

forecast_results = []

for pred in future_predictions:
    # Create feature vector in correct order
    feature_vector = [pred['features'].get(col, 0) for col in all_features]
    feature_vector_scaled = scaler.transform([feature_vector])
    
    # Predict
    predicted_demand = gb_model.predict(feature_vector_scaled)[0]
    
    forecast_results.append({
        'forecast_date': pred['forecast_date'],
        'product_category': pred['product_category'],
        'predicted_demand': max(0, int(predicted_demand))  # Ensure non-negative integer
    })

# Create forecast dataframe
forecast_results_df = pd.DataFrame(forecast_results)

print(f"✅ Predictions generated: {len(forecast_results_df):,} records")
print("\nSample forecasts:")
print(forecast_results_df.head(15))

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# SAVE FORECASTS TO DELTA TABLE
# ====================================================================
print("💾 Saving demand forecasts...\n")

# Convert to Spark DataFrame
forecast_spark_df = spark.createDataFrame(forecast_results_df)

# Write to gold schema
forecast_table = f"{CATALOG}.{GOLD_SCHEMA}.gold_demand_forecast"

(forecast_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(forecast_table))

print(f"✅ Saved forecasts to: {forecast_table}")
print(f"   Total forecasts: {len(forecast_results_df):,}")
print("=" * 80 + "\n")

In [0]:
print("\n" + "=" * 80)
print("🎉 DEMAND FORECASTING MODEL COMPLETE")
print("=" * 80)

print("\n📊 Model Performance Summary:")
print("  Random Forest:")
print(f"    - Test MAE: {mean_absolute_error(y_test, rf_model.predict(X_test_scaled)):.2f} units")
print(f"    - Test R²: {r2_score(y_test, rf_model.predict(X_test_scaled)):.4f}")

print("\n  Gradient Boosting:")
print(f"    - Test MAE: {mean_absolute_error(y_test, gb_model.predict(X_test_scaled)):.2f} units")
print(f"    - Test R²: {r2_score(y_test, gb_model.predict(X_test_scaled)):.4f}")

print("\n📁 Outputs Created:")
print(f"  - MLflow Experiment: {EXPERIMENT_NAME}")
print(f"  - Registered Models: retail_demand_random_forest, retail_demand_gradient_boosting")
print(f"  - Forecasts Table: {forecast_table}")

print("\n📈 Forecast Summary:")
total_predicted = forecast_results_df['predicted_demand'].sum()
print(f"  - Total predicted demand (next 3 months): {total_predicted:,} units")
print(f"  - Categories forecasted: {forecast_results_df['product_category'].nunique()}")

print("\n" + "=" * 80)
print("🎊 PHASE 4 - ML MODELS COMPLETE!")
print("=" * 80)
print("\n✅ All Models Built:")
print("  1. Churn Prediction (Classification)")
print("  2. Demand Forecasting (Regression)")
print("\n📝 Next Phase: Productionize with Model Serving & Workflows")
print("=" * 80 + "\n")

In [0]:
%sql
-- View demand forecasts
SELECT 
    forecast_date,
    product_category,
    predicted_demand
FROM workspace.retail_gold.gold_demand_forecast
ORDER BY forecast_date, predicted_demand DESC
LIMIT 30;

In [0]:
%sql
-- Top categories by predicted demand
SELECT 
    product_category,
    SUM(predicted_demand) as total_predicted_demand
FROM workspace.retail_gold.gold_demand_forecast
GROUP BY product_category
ORDER BY total_predicted_demand DESC
LIMIT 15;